<a href="https://colab.research.google.com/github/Marina4ij/dz3/blob/main/dz3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import pandas as pd
import os
import glob

# 1. Настройки
DB_NAME = "cicids2017.db"
CSV_FOLDER = "./CICIDS2017_CSVs" # Путь к папке, куда вы распаковали CSV файлы

# Создаем подключение
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# 2. Создание схемы базы данных
print("Создание таблиц...")

# Справочник типов атак (Labels)
cursor.execute('''
CREATE TABLE IF NOT EXISTS attack_labels (
    label_id INTEGER PRIMARY KEY AUTOINCREMENT,
    label_name TEXT UNIQUE NOT NULL,
    category TEXT NOT NULL, -- Например: 'Benign', 'DDoS', 'Web Attack', 'Bot'
    is_malicious BOOLEAN NOT NULL
)
''')

# Основная таблица сетевых потоков (Network Flows)
# Мы создадим её динамически на основе колонок из CSV,
# но добавим внешние ключи для связей.
cursor.execute('''
CREATE TABLE IF NOT EXISTS network_flows (
    flow_id TEXT PRIMARY KEY,
    timestamp TEXT,
    src_ip TEXT,
    src_port INTEGER,
    dst_ip TEXT,
    dst_port INTEGER,
    protocol TEXT,
    flow_duration REAL,
    total_fwd_packets INTEGER,
    total_bwd_packets INTEGER,
    label_id INTEGER,
    FOREIGN KEY (label_id) REFERENCES attack_labels(label_id)
)
''')
# Примечание: в реальном проекте таблица network_flows будет содержать все ~80 колонок.
# Pandas создаст недостающие колонки автоматически при первом же insert.

conn.commit()

# 3. Заполнение справочника атак (с учетом проблем кодировки в оригинальном датасете)
print("Заполнение справочника атак...")
labels_data = [
    ('BENIGN', 'Benign', False),
    ('DDoS', 'DDoS', True),
    ('PortScan', 'Reconnaissance', True),
    ('Bot', 'Botnet', True),
    ('Infiltration', 'Infiltration', True),
    ('Web Attack \x96 Brute Force', 'Web Attack', True), # Специфичный символ из оригинала
    ('Web Attack \x96 XSS', 'Web Attack', True),
    ('Web Attack \x96 Sql Injection', 'Web Attack', True),
    ('FTP-Patator', 'Brute Force', True),
    ('SSH-Patator', 'Brute Force', True),
    ('DoS slowloris', 'DoS', True),
    ('DoS Slowhttptest', 'DoS', True),
    ('DoS Hulk', 'DoS', True),
    ('DoS GoldenEye', 'DoS', True),
    ('Heartbleed', 'DoS', True)
]

# Очищаем названия от возможных артефактов кодировки перед вставкой
clean_labels = []
for name, cat, is_mal in labels_data:
    clean_name = name.replace('\x96', '-').replace('\xef\xbf\xbd', '-').strip()
    clean_labels.append((clean_name, cat, is_mal))

cursor.executemany('''
    INSERT OR IGNORE INTO attack_labels (label_name, category, is_malicious)
    VALUES (?, ?, ?)
''', clean_labels)
conn.commit()

# Создаем словарь для быстрого маппинга label_name -> label_id
cursor.execute("SELECT label_name, label_id FROM attack_labels")
label_mapping = {row[0]: row[1] for row in cursor.fetchall()}
# Добавляем варианты с разными тире на случай, если парсинг прошел иначе
label_mapping['Web Attack - Brute Force'] = label_mapping.get('Web Attack - Brute Force', 6)

# 4. Функция для очистки и загрузки CSV (Chunking для экономии RAM)
def load_cicids_csv(file_path):
    print(f"Обработка {os.path.basename(file_path)}...")

    # Читаем частями (chunk), чтобы не переполнить оперативную память
    chunk_size = 100000
    total_rows = 0

    for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
        # Очистка данных (критично для CICIDS2017!)
        # 1. Заменяем бесконечности и некорректные значения на NaN
        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk.fillna(0, inplace=True)

        # 2. Чистим названия атак от битых символов (артефакты кодировки)
        chunk['Label'] = chunk['Label'].apply(lambda x: str(x).replace('\x96', '-').replace('\xef\xbf\xbd', '-').strip())

        # 3. Маппим текстовые лейблы в ID из нашей БД
        chunk['label_id'] = chunk['Label'].map(label_mapping).fillna(1) # 1 - это BENIGN

        # 4. Оставляем только нужные колонки (или все, если хотите хранить 80 фич)
        # Для примера возьмем базовые + label_id. Если нужно все 80, просто уберите этот drop.
        cols_to_keep = [
            'Flow ID', 'Timestamp', 'Source IP', 'Source Port',
            'Destination IP', 'Destination Port', 'Protocol', 'Flow Duration',
            'Total Fwd Packets', 'Total Backward Packets', 'Label', 'label_id'
        ]

        # Фильтруем только существующие колонки (на случай опечаток в заголовках CSV)
        cols_to_keep = [c for c in cols_to_keep if c in chunk.columns]
        db_chunk = chunk[cols_to_keep]

        # Переименовываем колонки для соответствия БД (без пробелов)
        db_chunk.columns = [
            'flow_id', 'timestamp', 'src_ip', 'src_port', 'dst_ip', 'dst_port',
            'protocol', 'flow_duration', 'total_fwd_packets', 'total_bwd_packets',
            'label_name', 'label_id'
        ]

        # Загружаем в SQLite
        db_chunk.to_sql('network_flows', conn, if_exists='append', index=False, method='multi')
        total_rows += len(db_chunk)

    print(f"  -> Загружено строк: {total_rows}")

# 5. Запуск загрузки всех CSV из папки
import numpy as np # нужен для np.inf

if os.path.exists(CSV_FOLDER):
    csv_files = glob.glob(os.path.join(CSV_FOLDER, "*.csv"))
    if not csv_files:
        print(f"В папке {CSV_FOLDER} не найдено CSV файлов.")
    else:
        for file in csv_files:
            load_cicids_csv(file)
else:
    print(f"Папка {CSV_FOLDER} не найдена. Создайте её и скачайте туда CSV файлы CICIDS2017.")

# 6. Проверка и закрытие
print("\nСтатистика загрузки:")
cursor.execute("SELECT COUNT(*) FROM network_flows")
print(f"Всего сетевых потоков: {cursor.fetchone()[0]:,}")

cursor.execute('''
    SELECT al.label_name, al.category, COUNT(nf.flow_id) as flow_count
    FROM network_flows nf
    JOIN attack_labels al ON nf.label_id = al.label_id
    GROUP BY al.label_name
    ORDER BY flow_count DESC
''')
print("\nРаспределение трафика по типам:")
for row in cursor.fetchall():
    print(f"  {row[0]:<25} ({row[1]:<15}): {row[2]:>10,} потоков")

conn.close()
print(f"\n✓ База данных '{DB_NAME}' успешно создана и заполнена!")

Создание таблиц...
Заполнение справочника атак...
Папка ./CICIDS2017_CSVs не найдена. Создайте её и скачайте туда CSV файлы CICIDS2017.

Статистика загрузки:
Всего сетевых потоков: 0

Распределение трафика по типам:

✓ База данных 'cicids2017.db' успешно создана и заполнена!


CICIDS2017 / CSE-CIC-IDS2018 — стандартные датасеты от Канадского института кибербезопасности. Содержат нормальный сетевой трафик и множество видов атак (DDoS, брутфорс, SQL-инъекции и т.д.).
Где найти: Сайт UNB (Canadian Institute for Cybersecurity) или Kaggle.
CICIDS2017: https://www.unb.ca/cic/datasets/ids-2017.html (CSV-файлы с сетевым трафиком и атаками)